Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import xgboost as xgb
import warnings
warnings.filterwarnings("ignore")

Data Preprocessing  

B1

1 -> Loading dataset

In [10]:
df = pd.read_csv("Loan_default_dataset.csv")
print("Initial dataset shape:", df.shape)

Initial dataset shape: (255347, 18)


2 -> Basic Data Cleaning

In [11]:
df.drop_duplicates(inplace=True)

if 'LoanID' in df.columns:
    df.drop('LoanID', axis=1, inplace=True)

3 -> Identify Missing Values

In [12]:
print("\nMissing values before imputation:")
print(df.isnull().sum())


Missing values before imputation:
Age               0
Income            0
LoanAmount        0
CreditScore       0
MonthsEmployed    0
NumCreditLines    0
InterestRate      0
LoanTerm          0
DTIRatio          0
Education         0
EmploymentType    0
MaritalStatus     0
HasMortgage       0
HasDependents     0
LoanPurpose       0
HasCoSigner       0
Default           0
dtype: int64


4 -> Handling Missing Data

In [ ]:
numeric_features = ['Age', 'Income', 'LoanAmount', 'CreditScore', 
                    'MonthsEmployed', 'NumCreditLines', 'InterestRate', 
                    'LoanTerm', 'DTIRatio']
categorical_features = ['Education', 'EmploymentType', 'MaritalStatus', 
                        'HasMortgage', 'HasDependents', 'LoanPurpose', 'HasCoSigner']

binary_cols = ['HasMortgage', 'HasDependents', 'HasCoSigner']
for col in binary_cols:
    df[col] = df[col].map({'Yes': 1, 'No': 0})

num_imputer = SimpleImputer(strategy='median')
df[numeric_features] = num_imputer.fit_transform(df[numeric_features])

cat_cols = list(set(categorical_features) - set(binary_cols))
cat_imputer = SimpleImputer(strategy='most_frequent')
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

print("\nMissing values after imputation:")
print(df.isnull().sum())


Missing values after imputation:
Age               0
Income            0
LoanAmount        0
CreditScore       0
MonthsEmployed    0
NumCreditLines    0
InterestRate      0
LoanTerm          0
DTIRatio          0
Education         0
EmploymentType    0
MaritalStatus     0
HasMortgage       0
HasDependents     0
LoanPurpose       0
HasCoSigner       0
Default           0
dtype: int64


5 -> Feature Engineering

In [14]:
df['LoanToIncomeRatio'] = df['LoanAmount'] / df['Income']
df['FinancialRiskScore'] = df['CreditScore'] - df['LoanToIncomeRatio'] + (df['MonthsEmployed'] / 12)

numeric_features.extend(['LoanToIncomeRatio', 'FinancialRiskScore'])

6 -> Categorical Encoding

In [16]:
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

7 -> Scaling Numerical Features

In [15]:
scaler = StandardScaler()
df[numeric_features] = scaler.fit_transform(df[numeric_features])

for col in numeric_features:
    lower = df[col].quantile(0.01)
    upper = df[col].quantile(0.99)
    df[col] = np.clip(df[col], lower, upper)

In [17]:
print("\nFinal dataset shape:", df.shape)
print("\nFirst few rows of the preprocessed data:")
print(df.head())

df.to_csv("loan_default_preprocessed.csv", index=False)


Final dataset shape: (255347, 27)

First few rows of the preprocessed data:
        Age    Income  LoanAmount  CreditScore  MonthsEmployed  \
0  0.833990  0.089693   -1.086833    -0.341492        0.590533   
1  1.701221 -0.823021   -0.044309    -0.731666       -1.285731   
2  0.166888  0.043854    0.022715    -0.775718       -0.968209   
3 -0.767053 -1.303452   -1.168538     1.061875       -1.689849   
4  1.100830 -1.592855   -1.671921     0.369631       -1.487790   

   NumCreditLines  InterestRate  LoanTerm  DTIRatio  HasMortgage  ...  \
0        1.341937      0.261771 -0.001526 -0.260753            1  ...   
1       -1.343791     -1.308350  1.412793  0.778585            0  ...   
2        0.446694      1.156831 -0.708685 -0.823728            1  ...   
3        0.446694     -0.967805 -0.708685 -1.170174            0  ...   
4        1.341937     -1.052188  0.705634  0.995114            0  ...   

   LoanPurpose_Home  LoanPurpose_Other  MaritalStatus_Married  \
0             False   

Model Training  

B2

In [18]:
df = pd.read_csv("loan_default_preprocessed.csv")
print("Dataset shape:", df.shape)

X = df.drop("Default", axis=1)
y = df["Default"]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

Dataset shape: (255347, 27)
Training set shape: (204277, 26)
Testing set shape: (51070, 26)


In [19]:
xgb_model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)